In [ ]:
# ============================================================
# BIOMASSA DOS 3 ANOS DE UMA VEZ (Jucker et al. 2017, Eq. 7, angiospermas)
# AGB (kg) = 0.016 * (H * CD)^2.013 * exp(0.204^2 / 2)
# Altura de campo: vem do arquivo de 2026, pelo det_id (igual nos 3 anos).
# ============================================================
# Colab:  !pip install geopandas --quiet

import numpy as np
import pandas as pd
import geopandas as gpd

# ---------- Parâmetros ----------
ARQUIVOS = {
    2024: "/content/2024.gpkg",
    2025: "/content/2025.gpkg",
    2026: "/content/2026.gpkg",
}
ANO_ALTURA = 2026               # arquivo que contém a coluna de altura
COL_ID = "det_id"               # aparece como "ID" no QGIS
COL_ALTURA = "altura_m"
CRS_METROS = "EPSG:31980"       # SIRGAS 2000 / UTM 20S
EXCLUIR_IDS = []                # [] = todas as árvores


def agb_jucker(h, cd):
    return 0.016 * ((h * cd) ** 2.013) * np.exp(0.204 ** 2 / 2)       # kg


In [ ]:
# ============================================================
# 1) ALTURAS DE CAMPO (por det_id)
# ============================================================
ref = gpd.read_file(ARQUIVOS[ANO_ALTURA])
assert COL_ALTURA in ref.columns, f"{ANO_ALTURA}: falta a coluna '{COL_ALTURA}'."
alturas = pd.DataFrame({
    COL_ID: ref[COL_ID].astype(int),
    "altura_m": pd.to_numeric(ref[COL_ALTURA], errors="coerce"), #aqui (erros=) era pra caso não tivesse arvore no ano anterior e surgisse no ano seguinte, deve ser observado no estudo.
}).drop_duplicates(COL_ID)

In [ ]:
# ============================================================
# 2) CALCULAR CADA ANO
# ============================================================
tabelas = {}
for ano, caminho in ARQUIVOS.items():
    g = gpd.read_file(caminho).to_crs(CRS_METROS) ##caso o arquivo nao esteja configurato em metros
    g[COL_ID] = g[COL_ID].astype(int)
    #g = g[~g[COL_ID].isin(EXCLUIR_IDS)].copy() #caso eu fosse excluir uma arvore, mas ja consertei essa parte
    g = g.merge(alturas, on=COL_ID, how="left") #associando todas as alturas dos anos ao arquivo de 2026 ja que nao tenho alturas dos anos anteriores
    g = g.sort_values(COL_ID).reset_index(drop=True)

    sem_altura = g.loc[g["altura_m"].isna(), COL_ID].tolist()
    assert not sem_altura, f"{ano}: sem altura para os ids {sem_altura}"

    g["area_copa_m2"] = g.geometry.area
    g["cd_m"] = 2 * np.sqrt(g["area_copa_m2"] / np.pi)        # diâmetro equivalente
    #g["H_x_CD"] = g["altura_m"] * g["cd_m"] mudei a equação então não é necessário agora, mas posso utilizar no futuro para fins comparativos e gráficos
    g["agb_kg"] = agb_jucker(g["altura_m"], g["cd_m"])
    tabelas[ano] = g
    print(f"{ano}: {len(g)} árvores | biomassa total {g['agb_kg'].sum():.1f} kg")

# mesmos ids nos três anos?
conjuntos = [set(t[COL_ID]) for t in tabelas.values()]
assert all(c == conjuntos[0] for c in conjuntos), "Os ids não são os mesmos nos 3 anos."

2024: 18 árvores | biomassa total 799.9 kg
2025: 18 árvores | biomassa total 955.9 kg
2026: 18 árvores | biomassa total 873.2 kg


In [ ]:
# ============================================================
# 3) TABELA ÚNICA: UMA LINHA POR ÁRVORE
# ============================================================
largo = tabelas[ANO_ALTURA][[COL_ID, "altura_m"]].copy()
for ano, t in tabelas.items():
    parte = t[[COL_ID, "area_copa_m2", "cd_m", "agb_kg"]].rename(columns={
        "area_copa_m2": f"area_copa_{ano}_m2",
        "cd_m": f"cd_{ano}_m",
        "agb_kg": f"agb_{ano}_kg",
    })
    largo = largo.merge(parte, on=COL_ID, how="left")

anos = sorted(ARQUIVOS)
a0, a1 = anos[0], anos[-1]
largo[f"dAGB_{a0}_{a1}_kg"] = largo[f"agb_{a1}_kg"] - largo[f"agb_{a0}_kg"]
largo[f"V_AGB_{a0}_{a1}_pct"] = largo[f"dAGB_{a0}_{a1}_kg"] / largo[f"agb_{a0}_kg"] * 100

pd.set_option("display.width", 220)
print(largo.round(1).to_string(index=False))
print("\nBiomassa total (kg):", {a: round(largo[f"agb_{a}_kg"].sum(), 1) for a in anos})

 det_id  altura_m  area_copa_2024_m2  cd_2024_m  agb_2024_kg  area_copa_2025_m2  cd_2025_m  agb_2025_kg  area_copa_2026_m2  cd_2026_m  agb_2026_kg  dAGB_2024_2026_kg  V_AGB_2024_2026_pct
      0      10.1               92.7       10.9        209.2              127.9       12.8        289.0              117.5       12.2        265.4               56.2                 26.9
      1       7.5               67.1        9.2         83.0               69.2        9.4         85.6               58.4        8.6         72.1              -10.9                -13.1
      2       6.0               28.1        6.0         22.0               24.0        5.5         18.8               21.9        5.3         17.2               -4.8                -22.0
      3       6.0               30.0        6.2         23.6               28.5        6.0         22.4               23.2        5.4         18.2               -5.3                -22.7
      4       5.4               32.5        6.4         20.6     

In [ ]:
# ============================================================
# 4) SALVAR
# ============================================================
largo.to_csv("/content/biomassa_3anos.csv", index=False)
for ano, t in tabelas.items():
    t.to_file(f"/content/biomassa_{ano}.gpkg", driver="GPKG")

from google.colab import files
files.download("/content/biomassa_3anos.csv")
for ano in tabelas:
    files.download(f"/content/biomassa_{ano}.gpkg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**VARIAÇÃO DA ÁREA DE COPA (2024, 2025, 2026) — análise independente**
##Não usa altura nem biomassa.

In [ ]:
# ============================================================
# VARIAÇÃO DA ÁREA DE COPA (2024, 2025, 2026) — análise independente
# Não usa altura nem biomassa. Todas as árvores entram (sem exclusões).
# Usa a coluna "det_id" (que no QGIS aparece como "ID"): o mesmo número
# é a mesma árvore nos três anos.
# ============================================================
# Colab:  !pip install geopandas --quiet

import numpy as np
import pandas as pd
import geopandas as gpd

# ---------- Parâmetros ----------
ARQUIVOS = {
    2024: "/content/2024.gpkg",    # ajuste se os arquivos tiverem outro nome
    2025: "/content/2025.gpkg",    # (ex.: 2025_7b3225.gpkg)
    2026: "/content/2026.gpkg",
}
ANO_INICIAL, ANO_FINAL = 2024, 2026
CRS_METROS = "EPSG:31980"      # SIRGAS 2000 / UTM 20S (cálculo de área)
COL_ID = "det_id"              # aparece como "ID" na tabela do QGIS

LIMITE_BASE_M2 = 5.0           # área inicial abaixo disso: V% não confiável
TOLERANCIA_PCT = 10.0          # |V%| menor que isso = "estável" (ajuste/justifique)

# Ressalvas por árvore (aparecem no CSV e no GeoJSON do dashboard)
OBSERVACOES = {
    5: "possível poda",
}


# ============================================================
# 1) LER CADA ANO E CALCULAR A ÁREA DE CADA COPA
# ============================================================
areas = {}
geoms = {}
for ano, caminho in ARQUIVOS.items():
    g = gpd.read_file(caminho).to_crs(CRS_METROS)
    assert COL_ID in g.columns, f"{ano}: falta a coluna '{COL_ID}'."
    assert g[COL_ID].notna().all(), f"{ano}: há polígonos sem {COL_ID}."

    g[COL_ID] = g[COL_ID].astype(int)                      # det_id vem como texto
    g = g[[COL_ID, "geometry"]].dissolve(by=COL_ID, as_index=False)
    g[f"ca_{ano}"] = g.geometry.area                       # m²
    areas[ano] = g[[COL_ID, f"ca_{ano}"]]
    geoms[ano] = g[[COL_ID, "geometry"]]
    print(f"{ano}: {len(g)} árvores")


# ============================================================
# 2) TABELA ÚNICA: UMA LINHA POR ÁRVORE
# ============================================================
df = areas[ANO_INICIAL]
for ano in ARQUIVOS:
    if ano != ANO_INICIAL:
        df = df.merge(areas[ano], on=COL_ID, how="outer") #mesma coisa, nao descarta a arvore apenas preenche nulo (how Outer join)
df = df.sort_values(COL_ID).reset_index(drop=True)

ausentes = df[df[[f"ca_{a}" for a in ARQUIVOS]].isna().any(axis=1)]
print("\nÁrvores ausentes em algum ano:", ausentes[COL_ID].tolist())
print("Total de árvores na tabela:", len(df))


# ============================================================
# 3) VARIAÇÃO DA ÁREA DE COPA
#    ΔCA = CA_final − CA_inicial        V(%) = ΔCA / CA_inicial × 100
# ============================================================
def variacao(df, a0, a1):
    d = df[f"ca_{a1}"] - df[f"ca_{a0}"]
    df[f"dCA_{a0}_{a1}_m2"] = d
    df[f"V_{a0}_{a1}_pct"] = d / df[f"ca_{a0}"] * 100

variacao(df, 2024, 2026)      # indicador principal
variacao(df, 2024, 2025)
variacao(df, 2025, 2026)

# Sinalizações
df["base_pequena"] = df[f"ca_{ANO_INICIAL}"] < LIMITE_BASE_M2

col_v = f"V_{ANO_INICIAL}_{ANO_FINAL}_pct"
df["tendencia"] = np.select(
    [df[col_v] > TOLERANCIA_PCT, df[col_v] < -TOLERANCIA_PCT],
    ["aumento", "redução"],
    default="estável",
)
df.loc[df["base_pequena"] | df[col_v].isna(), "tendencia"] = "indeterminada"

df["observacao"] = df[COL_ID].map(OBSERVACOES).fillna("")

pd.set_option("display.width", 200)
print(df[[COL_ID, "ca_2024", "ca_2025", "ca_2026",
          f"dCA_{ANO_INICIAL}_{ANO_FINAL}_m2", col_v, "tendencia"]]
      .round(1).to_string(index=False))

print("\nÁrea total de copa (m²):",
      {a: round(df[f"ca_{a}"].sum(), 1) for a in ARQUIVOS})
print(df["tendencia"].value_counts().to_string())


# ============================================================
# 4) ALTURA E BIOMASSA (altura de campo do arquivo de 2026)
#    AGB (kg) = 0.016 * (H * CD)^2.013 * exp(0.204^2 / 2)   (Jucker 2017, Eq. 7)
#    Serve só para o popup do mapa; a análise de copa não depende disso.
#      ###############PARA O DASHBOARD)###############
# ============================================================
ANO_ALTURA = 2026
COL_ALTURA = "altura_m"

ref = gpd.read_file(ARQUIVOS[ANO_ALTURA])
assert COL_ALTURA in ref.columns, f"{ANO_ALTURA}: falta a coluna '{COL_ALTURA}'."
alturas = pd.DataFrame({
    COL_ID: ref[COL_ID].astype(int),
    "altura_m": pd.to_numeric(ref[COL_ALTURA], errors="coerce"),
}).drop_duplicates(COL_ID)

df = df.merge(alturas, on=COL_ID, how="left")
sem_altura = df.loc[df["altura_m"].isna(), COL_ID].tolist()
assert not sem_altura, f"Sem altura para os ids: {sem_altura}"


# ============================================================
# 5) EXPORTAR PARA O DASHBOARD
# ============================================================
df.to_csv("/content/variacao_copa.csv", index=False)

# Um único GeoJSON com os 3 anos: uma linha por árvore E ano
# (18 árvores x 3 anos = 54 polígonos), com a coluna "ano" para filtrar no mapa
partes = []
for ano in ARQUIVOS:
    p = geoms[ano].merge(df, on=COL_ID, how="left")
    p["ano"] = ano
    p["area_copa_m2"] = p[f"ca_{ano}"]                        # área daquele ano
    p["cd_m"] = 2 * np.sqrt(p["area_copa_m2"] / np.pi)        # diâmetro equivalente   #####AQUI UTILIZO OS MESMOS CALCULOS DE ANTES###
    p["biomassa_kg"] = (0.016 * ((p["altura_m"] * p["cd_m"]) ** 2.013)
                        * np.exp(0.204 ** 2 / 2))
    partes.append(p)

mapa = pd.concat(partes, ignore_index=True)
mapa = gpd.GeoDataFrame(mapa, geometry="geometry", crs=CRS_METROS).to_crs("EPSG:4326")
mapa.to_file("/content/arvores_3anos.geojson", driver="GeoJSON")
print(f"\nGeoJSON: {len(mapa)} polígonos (esperado: 54)")
print("Colunas do GeoJSON:", [c for c in mapa.columns if c != "geometry"])

from google.colab import files
files.download("/content/variacao_copa.csv")
files.download("/content/arvores_3anos.geojson")

2024: 18 árvores
2025: 18 árvores
2026: 18 árvores

Árvores ausentes em algum ano: []
Total de árvores na tabela: 18
 det_id  ca_2024  ca_2025  ca_2026  dCA_2024_2026_m2  V_2024_2026_pct     tendencia
      0     92.7    127.9    117.5              24.7             26.7       aumento
      1     67.1     69.2     58.4              -8.8            -13.1       redução
      2     28.1     24.0     21.9              -6.1            -21.9       redução
      3     30.0     28.5     23.2              -6.8            -22.5       redução
      4     32.5     32.6     27.9              -4.6            -14.2       redução
      5     38.2     32.2     32.2              -6.1            -15.8       redução
      6     33.6     34.9     17.3             -16.2            -48.4       redução
      7      6.3     10.5     11.7               5.5             86.8       aumento
      8     31.1     25.0     23.9              -7.3            -23.4       redução
      9      1.3      8.7     11.5         

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>